# Task 10: Deploying the Model as a Live Web App

**NeuroFive ML Track, Week 5**

Every model built so far has lived inside a notebook. Nobody outside this notebook can use them. This task closes that gap by turning the Task 7 Titanic pipeline into a web app that anyone can open in a browser.

**Which model, and why**

The Task 7 pipeline is the right choice here, and not only because it scored well. It takes **raw passenger details** as input and handles imputation, feature scaling, and categorical encoding internally. That means the web app can pass user input straight in without reproducing any preprocessing logic, which is exactly what a pipeline is for.

The alternatives are worse fits. The Task 9 fraud model needs 28 anonymized PCA components as input, which no human could sensibly type in. The Task 6 churn model needs 19 fields. Titanic needs 8, and they are all things a person can understand.

**What this notebook produces**

1. `titanic_pipeline.joblib`, the trained model artifact
2. `app.py`, the Streamlit application
3. `requirements.txt`, with versions pinned to this exact environment
4. Deployment instructions for Streamlit Community Cloud

In [1]:
import sys
import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("joblib:", joblib.__version__)

Python: 3.13.15
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.1.3
joblib: 1.5.3


Those version numbers matter more than they look. A model saved by one version of scikit-learn may refuse to load, or load incorrectly, under a different version. Section 5 uses these exact numbers to build `requirements.txt`, which is the single most common reason a deployment that works locally fails once it is online.

## 1. Rebuilding and Training the Model

This repeats the Task 7 pipeline. The model is retrained here so this notebook produces its own artifact rather than depending on a file created elsewhere.

In [2]:
url = "https://raw.githubusercontent.com/BadarRao/neurofive-ml-track/main/train.csv"
df = pd.read_csv(url)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 891
Columns: 12


In [3]:
def engineer_features(data):
    """Create engineered features. All operations are row-wise."""
    data = data.copy()
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    common_titles = ['Mr', 'Mrs', 'Miss', 'Master']
    data['Title'] = data['Title'].apply(lambda t: t if t in common_titles else 'Rare')
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    return data


df_engineered = engineer_features(df)

print("Titles found:")
print(df_engineered['Title'].value_counts())

Titles found:
Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64


In [4]:
FEATURE_COLUMNS = [
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
    'FamilySize', 'IsAlone', 'Title', 'FarePerPerson'
]

numerical_features = [
    'Pclass', 'Age', 'SibSp', 'Parch', 'Fare',
    'FamilySize', 'IsAlone', 'FarePerPerson'
]
categorical_features = ['Sex', 'Embarked', 'Title']

X = df_engineered[FEATURE_COLUMNS]
y = df_engineered['Survived']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numerical', numerical_transformer, numerical_features),
    ('categorical', categorical_transformer, categorical_features)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

print("Pipeline built with", len(FEATURE_COLUMNS), "input features")

Pipeline built with 11 input features


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print("Test set accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Cross validated accuracy:", round(cv_scores.mean(), 4),
      "with standard deviation", round(cv_scores.std(), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

Test set accuracy: 0.8436
Cross validated accuracy: 0.8283 with standard deviation 0.0078

                 precision    recall  f1-score   support

Did Not Survive       0.85      0.90      0.88       110
       Survived       0.83      0.75      0.79        69

       accuracy                           0.84       179
      macro avg       0.84      0.83      0.83       179
   weighted avg       0.84      0.84      0.84       179



## 2. Retraining on the Full Dataset

For deployment, the model is refitted on **all 891 passengers** rather than only the 712 in the training split.

The reason is that the split existed to give an honest performance estimate, and that estimate has now been taken. Holding data back at deployment time would mean shipping a model trained on less information than is available, for no benefit. The accuracy figures reported to users still come from the held out evaluation above.

In [6]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

final_pipeline.fit(X, y)

print("Final model trained on all", len(X), "passengers")

Final model trained on all 891 passengers


## 3. Saving the Model

`joblib` stores the whole fitted pipeline: the imputer medians, the scaler statistics, the encoder categories, and the model coefficients. Saving only the classifier would be useless, because raw user input could not be transformed to match what it expects.

In [7]:
MODEL_PATH = 'titanic_pipeline.joblib'

joblib.dump(final_pipeline, MODEL_PATH)

import os
size_kb = os.path.getsize(MODEL_PATH) / 1024

print("Saved to:", MODEL_PATH)
print("File size:", round(size_kb, 1), "KB")

Saved to: titanic_pipeline.joblib
File size: 5.0 KB


## 4. Testing the Saved Model Before Deploying

Worth doing here rather than discovering a problem after deployment. This reloads the file as a fresh object and runs the exact call the app will make.

In [8]:
loaded = joblib.load(MODEL_PATH)

# A test passenger, entered the way the app will construct one
test_passenger = pd.DataFrame([{
    'Pclass': 3,
    'Sex': 'male',
    'Age': 30.0,
    'SibSp': 0,
    'Parch': 0,
    'Fare': 7.25,
    'Embarked': 'S',
    'Title': 'Mr'
}])

# Rebuild the engineered features the same way the app does
test_passenger['FamilySize'] = test_passenger['SibSp'] + test_passenger['Parch'] + 1
test_passenger['IsAlone'] = (test_passenger['FamilySize'] == 1).astype(int)
test_passenger['FarePerPerson'] = test_passenger['Fare'] / test_passenger['FamilySize']

pred = loaded.predict(test_passenger[FEATURE_COLUMNS])[0]
prob = loaded.predict_proba(test_passenger[FEATURE_COLUMNS])[0][1]

print("Third class adult male travelling alone")
print("  Prediction:", "Survived" if pred == 1 else "Did not survive")
print("  Survival probability:", f"{prob:.2%}")

Third class adult male travelling alone
  Prediction: Did not survive
  Survival probability: 7.56%


In [9]:
# A contrasting passenger, to confirm the model responds to input
test_passenger_2 = pd.DataFrame([{
    'Pclass': 1,
    'Sex': 'female',
    'Age': 28.0,
    'SibSp': 1,
    'Parch': 1,
    'Fare': 120.0,
    'Embarked': 'C',
    'Title': 'Mrs'
}])

test_passenger_2['FamilySize'] = test_passenger_2['SibSp'] + test_passenger_2['Parch'] + 1
test_passenger_2['IsAlone'] = (test_passenger_2['FamilySize'] == 1).astype(int)
test_passenger_2['FarePerPerson'] = test_passenger_2['Fare'] / test_passenger_2['FamilySize']

pred2 = loaded.predict(test_passenger_2[FEATURE_COLUMNS])[0]
prob2 = loaded.predict_proba(test_passenger_2[FEATURE_COLUMNS])[0][1]

print("First class woman travelling with family")
print("  Prediction:", "Survived" if pred2 == 1 else "Did not survive")
print("  Survival probability:", f"{prob2:.2%}")

First class woman travelling with family
  Prediction: Survived
  Survival probability: 95.94%


## 5. Generating requirements.txt

**This is the step that most often breaks a deployment.**

Streamlit Community Cloud builds a fresh environment from `requirements.txt`. If it installs a different scikit-learn version than the one that created the `.joblib` file, loading the model can fail outright or, worse, succeed while behaving incorrectly.

The cell below writes the file using the versions actually running right now, so the deployed environment matches the one that trained the model.

In [10]:
requirements = f"""streamlit
scikit-learn=={sklearn.__version__}
pandas=={pd.__version__}
numpy=={np.__version__}
joblib=={joblib.__version__}
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("requirements.txt written:")
print()
print(requirements)

requirements.txt written:

streamlit
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.1.3
joblib==1.5.3



`streamlit` is deliberately left unpinned, since the app uses only stable, widely available components and benefits from picking up fixes. Everything involved in loading or running the model is pinned exactly.

## 6. Writing the Streamlit App

The cell below writes `app.py` to disk. In Colab, the file appears in the file panel on the left and can be downloaded from there.

**How the app works**

- `@st.cache_resource` loads the model once and keeps it in memory, instead of reloading on every interaction.
- Input widgets collect the eight fields a user can reasonably provide.
- `engineer_features` recreates `FamilySize`, `IsAlone`, and `FarePerPerson`. This has to live in the app because those features were built outside the pipeline in Task 7.
- The pipeline handles everything else: imputation, scaling, and encoding.
- `predict_proba` supplies the probability, which is more informative to a user than a bare yes or no.

**Note on `Title`:** the app offers a dropdown rather than asking for a full name. The training code extracted the title with a regex over the `Name` column, and asking a user to type a period correct Edwardian name just so a regex can parse it would be poor design. The dropdown supplies the same value directly.

In [11]:
%%writefile app.py
"""
Titanic Survival Predictor
NeuroFive ML Track, Task 10

A Streamlit web app that loads the trained pipeline from Task 7
and predicts survival for passenger details entered by the user.
"""

import streamlit as st
import pandas as pd
import joblib

# ----------------------------------------------------------------------
# Page configuration
# ----------------------------------------------------------------------

st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢",
    layout="centered"
)

MODEL_PATH = "titanic_pipeline.joblib"

# The exact column order the pipeline was fitted on.
# This must match the training notebook or the prediction will fail.
FEATURE_COLUMNS = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "FamilySize", "IsAlone", "Title", "FarePerPerson"
]


# ----------------------------------------------------------------------
# Model loading
# ----------------------------------------------------------------------

@st.cache_resource
def load_model():
    """Load the saved pipeline once and keep it in memory across reruns."""
    return joblib.load(MODEL_PATH)


try:
    model = load_model()
except FileNotFoundError:
    st.error(
        f"Could not find `{MODEL_PATH}`. Make sure the model file sits in the "
        "same folder as this app and has been committed to the repository."
    )
    st.stop()
except Exception as error:
    st.error(
        "The model file could not be loaded. This is usually a library version "
        "mismatch between the environment that trained the model and this one. "
        "Check that requirements.txt pins the same scikit-learn version used "
        "during training."
    )
    st.exception(error)
    st.stop()


# ----------------------------------------------------------------------
# Feature engineering
# ----------------------------------------------------------------------

def engineer_features(data):
    """
    Recreate the engineered features from Task 7.

    These were built outside the pipeline, so they have to be rebuilt here
    before calling predict. All operations are row-wise.
    """
    data = data.copy()
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    data["FarePerPerson"] = data["Fare"] / data["FamilySize"]
    return data


# ----------------------------------------------------------------------
# Header
# ----------------------------------------------------------------------

st.title("🚢 Titanic Survival Predictor")

st.write(
    "Enter passenger details below and the model will estimate their chance of "
    "surviving the Titanic disaster. The model is a Logistic Regression pipeline "
    "trained on the Kaggle Titanic dataset."
)

st.divider()


# ----------------------------------------------------------------------
# Input form
# ----------------------------------------------------------------------

st.subheader("Passenger Details")

col1, col2 = st.columns(2)

with col1:
    pclass = st.selectbox(
        "Ticket Class",
        options=[1, 2, 3],
        index=2,
        format_func=lambda x: {1: "1st Class", 2: "2nd Class", 3: "3rd Class"}[x],
        help="Passenger class. First class cabins were closer to the lifeboats."
    )

    sex = st.radio("Sex", options=["male", "female"], horizontal=True)

    age = st.slider("Age", min_value=0, max_value=80, value=30)

    title = st.selectbox(
        "Title",
        options=["Mr", "Mrs", "Miss", "Master", "Rare"],
        help="Taken from the passenger's name. 'Master' indicates a young boy. "
             "'Rare' covers titles such as Dr, Rev, Col and Countess."
    )

with col2:
    sibsp = st.number_input(
        "Siblings / Spouses aboard",
        min_value=0, max_value=10, value=0, step=1
    )

    parch = st.number_input(
        "Parents / Children aboard",
        min_value=0, max_value=10, value=0, step=1
    )

    fare = st.number_input(
        "Ticket Fare (£)",
        min_value=0.0, max_value=550.0, value=32.0, step=1.0,
        help="Total fare paid for the ticket, not per person."
    )

    embarked = st.selectbox(
        "Port of Embarkation",
        options=["S", "C", "Q"],
        format_func=lambda x: {
            "S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"
        }[x]
    )

# Show the derived values so the user can see what the model actually receives
family_size = sibsp + parch + 1
st.caption(
    f"Derived automatically: family size {family_size}, "
    f"{'travelling alone' if family_size == 1 else 'travelling with family'}, "
    f"fare per person £{fare / family_size:.2f}"
)

st.divider()


# ----------------------------------------------------------------------
# Prediction
# ----------------------------------------------------------------------

if st.button("Predict Survival", type="primary", width='stretch'):

    passenger = pd.DataFrame([{
        "Pclass": pclass,
        "Sex": sex,
        "Age": float(age),
        "SibSp": int(sibsp),
        "Parch": int(parch),
        "Fare": float(fare),
        "Embarked": embarked,
        "Title": title
    }])

    passenger = engineer_features(passenger)

    prediction = model.predict(passenger[FEATURE_COLUMNS])[0]
    probability = model.predict_proba(passenger[FEATURE_COLUMNS])[0][1]

    st.subheader("Prediction")

    if prediction == 1:
        st.success(f"**Likely to survive** with {probability:.1%} estimated probability")
    else:
        st.error(f"**Unlikely to survive** with {probability:.1%} estimated survival probability")

    st.progress(float(probability))

    result_col1, result_col2 = st.columns(2)
    result_col1.metric("Survival probability", f"{probability:.1%}")
    result_col2.metric("Model verdict", "Survived" if prediction == 1 else "Did not survive")

    with st.expander("What the model received"):
        st.dataframe(passenger[FEATURE_COLUMNS], width='stretch')


# ----------------------------------------------------------------------
# Footer
# ----------------------------------------------------------------------

st.divider()

with st.expander("About this model"):
    st.markdown(
        """
        **Model:** Logistic Regression inside a scikit-learn Pipeline

        **Preprocessing handled automatically by the pipeline:**
        - Median imputation for missing numerical values
        - Most frequent imputation for missing categorical values
        - StandardScaler on numerical features
        - OneHotEncoder on categorical features

        **Engineered features:** FamilySize, IsAlone, Title, FarePerPerson

        **Performance:** roughly 0.83 cross validated accuracy on the training data,
        with about 0.84 accuracy on a held out test set.

        This is a learning project built for the NeuroFive ML Track. The predictions
        describe patterns in the 1912 passenger manifest and are not meaningful for
        anything beyond that dataset.
        """
    )

Writing app.py


In [12]:
# Confirm the file is valid Python before deploying it
import py_compile

py_compile.compile('app.py', doraise=True)

print("app.py compiles without syntax errors")

app.py compiles without syntax errors


## 7. Files to Commit

Three files need to be in the repository, and Streamlit Cloud expects them at the paths given in the deploy settings.

| File | Purpose |
|---|---|
| `app.py` | The Streamlit application |
| `titanic_pipeline.joblib` | The trained model |
| `requirements.txt` | Pinned dependencies for the build |

In Colab, download all three from the file panel on the left, then commit them to the repository.

In [13]:
import os

for filename in ['app.py', 'titanic_pipeline.joblib', 'requirements.txt']:
    if os.path.exists(filename):
        size = os.path.getsize(filename)
        unit = 'KB' if size < 1024 * 1024 else 'MB'
        divisor = 1024 if unit == 'KB' else 1024 * 1024
        print(f"  {filename:<28} {size / divisor:>8.1f} {unit}")
    else:
        print(f"  {filename:<28} MISSING")

  app.py                            6.8 KB
  titanic_pipeline.joblib           5.0 KB
  requirements.txt                  0.1 KB


## 8. Deploying to Streamlit Community Cloud

**Step 1.** Commit `app.py`, `titanic_pipeline.joblib`, and `requirements.txt` to the repository root.

**Step 2.** Go to `share.streamlit.io` and sign in with GitHub.

**Step 3.** Click **New app**, then select:

- Repository: `BadarRao/neurofive-ml-track`
- Branch: `main`
- Main file path: `app.py`

**Step 4.** Click **Deploy**. The first build takes a few minutes while dependencies install.

**Step 5.** Copy the resulting URL, which looks like `https://your-app-name.streamlit.app`, and add it to the README.

### Testing the app locally first

Running it locally before deploying catches problems faster than waiting on cloud builds:

```bash
pip install -r requirements.txt
streamlit run app.py
```

It opens at `http://localhost:8501`.

### If the deployment fails

| Symptom | Likely cause |
|---|---|
| `FileNotFoundError` on the joblib file | The model was not committed, or `app.py` is not in the same folder |
| `InconsistentVersionWarning` or an unpickling error | `requirements.txt` pins a different scikit-learn version than the one that trained the model |
| Build times out or fails installing packages | A pinned version does not exist for the Python version Streamlit Cloud is using. Loosen that pin and redeploy |
| App loads but predictions error | Column names or their order do not match `FEATURE_COLUMNS` from training |

The Streamlit Cloud interface shows full build logs, which name the failing package directly.

## 9. Summary

**What was built**

1. Retrained the Task 7 pipeline and refitted it on all 891 passengers for deployment.
2. Saved the complete pipeline, preprocessing included, with joblib.
3. Verified the saved artifact by reloading it and predicting on two contrasting passengers.
4. Generated `requirements.txt` with versions pinned to the training environment.
5. Wrote and syntax checked a Streamlit app that collects raw input and returns a probability.

**Why the pipeline made this straightforward**

The app contains no imputation code, no scaling code, and no encoding code. It collects eight fields, derives three more, and calls `predict`. Every transformation applied during training is reapplied automatically and in the same order. Had preprocessing been done manually across notebook cells, as in Tasks 2 and 3, all of that logic would have to be rewritten inside the app and kept in sync by hand, which is exactly where deployment bugs come from.

**Limitations**

- The app serves one prediction at a time. A production service would expose a batch endpoint.
- No input validation beyond the widget ranges. A user can specify 10 siblings and an age of 0.
- The model is loaded from a file in the repository. Real deployments use a model registry with versioning and rollback.
- The 0.5 decision threshold is the default. Task 9 showed why that is worth choosing deliberately.